In [12]:
from src.core.services.connection.databricks_connector import DatabricksConnectionService

conn = DatabricksConnectionService(app_name="yandex-data-batch-ingestion")
conn.connect()

spark = conn.spark

print(" Connected to Spark!")
print(f"Spark Version: {spark.version}")

2026-10-02 11:39:40.910 | INFO     | src.core.services.connection.databricks_connector:connect:36 - SparkSession ready | app=yandex-data-batch-ingestion | version=4.2.0 | runtime=serverless-connect


 Connected to Spark!
Spark Version: 4.2.0


In [13]:
import os

catalog = os.environ.get("DATABRICKS_CATALOG", "yandex_go_dev")
raw_schema = f"{catalog}.raw_files"
volume_name = "landing"

print(f"🔧 Checking infrastructure for catalog [{catalog}]...")


volume_path = f"/Volumes/{catalog}/raw_files/{volume_name}"
print(f" Volume is ready at: {volume_path}")

🔧 Checking infrastructure for catalog [yandex_go_dev]...
 Volume is ready at: /Volumes/yandex_go_dev/raw_files/landing


In [14]:
from src.core.services.data_ingestion import UserBatchIngestionService, TaxiTripBatchIngestionService

user_service = UserBatchIngestionService()
user_service.run(spark)

taxi_service = TaxiTripBatchIngestionService()
taxi_service.run(spark)

2026-10-02 11:39:40.924 | INFO     | src.core.services.data_ingestion.base:run:77 - Start data ingestion for yandex_go_dev.bronze.users
2026-10-02 11:39:40.925 | INFO     | src.core.services.data_ingestion.base:extract:41 - Reading data [avro] from: /Volumes/yandex_go_dev/raw_files/landing/avro
2026-10-02 11:39:40.925 | INFO     | src.core.services.data_ingestion.base:load:61 - Writing data in table Delta: yandex_go_dev.bronze.users
2026-10-02 11:39:46.536 | INFO     | src.core.services.data_ingestion.base:load:74 - Successfully loaded in yandex_go_dev.bronze.users
2026-10-02 11:39:46.538 | INFO     | src.core.services.data_ingestion.base:run:77 - Start data ingestion for yandex_go_dev.bronze.taxi
2026-10-02 11:39:46.539 | INFO     | src.core.services.data_ingestion.base:extract:41 - Reading data [parquet] from: /Volumes/yandex_go_dev/raw_files/landing/parquet
2026-10-02 11:39:46.541 | INFO     | src.core.services.data_ingestion.base:load:61 - Writing data in table Delta: yandex_go_dev

In [15]:
users_count = spark.table(f"{catalog}.bronze.users").count()
taxi_count = spark.table(f"{catalog}.bronze.taxi").count()

print(f"📊 Count users in bronze.users: {users_count}")
print(f"📊 Count of trips in bronze.taxi: {taxi_count}")


display(
    spark.table(f"{catalog}.bronze.taxi").orderBy(
        "_ingested_at", ascending=False
    )
)

📊 Count users in bronze.users: 500
📊 Count of trips in bronze.taxi: 6000


DataFrame[id: string, user_id: string, driver_id: string, vendor_id: int, pickup_datetime: timestamp, dropoff_datetime: timestamp, passenger_count: bigint, trip_distance: double, rate_code_id: bigint, store_and_fwd_flag: string, pu_location_id: int, do_location_id: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, airport_fee: double, _rescued_data: string, _ingested_at: timestamp, _source_file: string]